In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display, Image, HTML
import random

# Nhãn cử chỉ
class_names = ['Hello', 'IloveYou', 'No', 'Please', 'Thanks', 'Yes']

# Đường dẫn ảnh test
test_dir = './Data/test/images'  # Thay nếu thư mục khác

# Load mô hình YOLO
model = YOLO('best.pt')  # Thay nếu mô hình ở nơi khác

# Tạo thư mục lưu ảnh định tính
correct_dir = './output/correct'
incorrect_dir = './output/incorrect'
for cname in class_names:
    os.makedirs(os.path.join(correct_dir, cname), exist_ok=True)
    os.makedirs(os.path.join(incorrect_dir, cname), exist_ok=True)

# Hàm làm nhiễu ảnh
def distort_image(image):
    # Làm mờ
    blurred = cv2.GaussianBlur(image, (9, 9), 0)

    # Xoay nhẹ
    h, w = blurred.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), 15, 1.0)
    rotated = cv2.warpAffine(blurred, M, (w, h))

    # Thêm noise
    noise = np.random.normal(0, 25, rotated.shape).astype(np.uint8)
    distorted = cv2.add(rotated, noise)

    return distorted

# Chuẩn bị dữ liệu
y_true = []
y_pred = []

# Chọn ngẫu nhiên 30% ảnh sẽ bị nhiễu
all_filenames = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
distort_count = int(0.3 * len(all_filenames))
distorted_set = set(random.sample(all_filenames, distort_count))

# Duyệt qua ảnh test
for fname in all_filenames:
    true_label = next((c for c in class_names if fname.lower().startswith(c.lower())), None)
    if true_label is None:
        continue

    img_path = os.path.join(test_dir, fname)
    img = cv2.imread(img_path)
    if img is None:
        continue

    # Gây nhiễu nếu nằm trong distorted_set
    input_img = distort_image(img) if fname in distorted_set else img

    results = model.predict(source=input_img, conf=0.25, verbose=False)
    pred_label = (
        results[0].names[int(results[0].boxes.cls[0])]
        if results[0].boxes else "No Detection"
    )

    y_true.append(true_label)
    y_pred.append(pred_label)

    # Annotate ảnh để lưu
    annotated_img = results[0].plot()
    save_path = os.path.join(
        correct_dir if pred_label == true_label else incorrect_dir,
        true_label,
        fname
    )
    cv2.imwrite(save_path, annotated_img)

# In báo cáo định lượng
print("=== Classification Report ===")
print(classification_report(
    y_true, y_pred, labels=class_names, target_names=class_names, zero_division=0
))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=class_names)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('./output/confusion_matrix.png')
plt.show()

# Hiển thị ảnh định tính
def show_images(folder, label, count=3):
    folder_path = os.path.join(folder, label)
    if not os.path.exists(folder_path):
        print(f'Không tìm thấy thư mục {folder_path}')
        return

    files = os.listdir(folder_path)[:count]
    images_html = ''
    for f in files:
        image_path = os.path.join(folder_path, f)
        images_html += f'<img src="files/{image_path}" width="200" style="margin: 5px;">'
    display(HTML(images_html))

for gesture in class_names:
    print(f'✅ Ví dụ đúng: {gesture}')
    show_images(correct_dir, gesture)

    print(f'❌ Ví dụ sai: {gesture}')
    show_images(incorrect_dir, gesture)
